# Analisis de marcha en Free-Living (ficheros .cwa)
Este notebook es independiente del script principal, pero **reutiliza sus funciones** importandolas desde `gait_lib.py` (modulo compartido).

A diferencia del dataset con protocolo, aqui no hay `patients.csv` con actividades etiquetadas -- se analiza la grabacion completa de cada paciente. Los ficheros son `.cwa` (formato binario de Axivity), que se leen con `skdh.io.ReadCwa` en vez de `pandas.read_csv`.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install -q scikit-digital-health


In [ ]:
import os
import sys
import pandas as pd

# Carpeta donde guardaste gait_lib.py (ajusta si lo pones en otro sitio)
sys.path.append('/content/drive/MyDrive/dataset')

from gait_lib import build_gait_old, find_patient_files, run_gait_freeliving


## 2. Configuracion

In [ ]:
FREE_LIVING_DIR = "/content/drive/MyDrive/dataset_pd/free_living"
OUTPUT_DIR       = "/content/drive/MyDrive/Lasige/FreeLiving_PD"
os.makedirs(OUTPUT_DIR, exist_ok=True)

GROUP = "PD"          # cambia a "Control" si corres la carpeta de free_living de controles
MAX_PATIENTS = 15     # limite para pruebas; sube el numero cuando confirmes que funciona
PATIENT_HEIGHT_M = 1.69


## 3. Localizar los ficheros de trunk (.cwa)

Los nombres de fichero no son 100% consistentes (`trunk-`, `trunkfreelivi`, `trunl-`...), asi que primero listamos que se encontro para revisarlo antes de correr nada pesado.

In [ ]:
trunk_files, skipped, posibles_erratas = find_patient_files(
    FREE_LIVING_DIR, keyword="trunk", fuzzy_keyword="trun"
)

print(f"Pacientes con fichero de trunk detectado: {len(trunk_files)}")
for pid, path in list(trunk_files.items())[:10]:
    print(f"  {pid} -> {os.path.basename(path)}")

# Aviso si algun 'trunk' encontrado NO es .cwa (por si hay formatos mezclados en la carpeta)
no_cwa = {pid: p for pid, p in trunk_files.items() if not p.lower().endswith('.cwa')}
if no_cwa:
    print(f"\n[AVISO] {len(no_cwa)} fichero(s) de trunk NO son .cwa (se leeran como .csv):")
    for pid, p in no_cwa.items():
        print(f"  {pid} -> {os.path.basename(p)}")

print(f"\nFicheros ignorados (no contienen 'trunk'): {len(skipped)}")

if posibles_erratas:
    print("\n[REVISAR] Estos ficheros contienen 'trun' pero no 'trunk' "
          "(posibles erratas de nombre, ej. 'trunl-'):")
    for name in posibles_erratas:
        print(f"  {name}")


Si en la celda anterior aparecen nombres en **[REVISAR]**, decide que hacer con ellos (renombrarlos en Drive, o anadirlos manualmente a `trunk_files` abajo) antes de continuar.

```python
# Ejemplo para anadir manualmente un fichero con nombre atipico:
# trunk_files["30837"] = "/content/drive/MyDrive/dataset_pd/free_living/30837-trunl-....cwa"
```

## 4. (Opcional) Probar la lectura de UN fichero .cwa antes de correr todo

Los `.cwa` pueden ser grabaciones de varios dias -- vale la pena confirmar que se leen bien y ver cuantas horas de datos tienen antes de lanzar el analisis completo.

In [ ]:
from gait_lib import read_recording

test_pid = list(trunk_files.keys())[0]
t, acc, recording_start = read_recording(trunk_files[test_pid])
print(f"Paciente {test_pid}")
print(f"Inicio de grabacion: {recording_start}")
print(f"Duracion: {t[-1]/3600:.1f} horas")
print(f"Muestras: {len(t)}")


## 5. Correr el analisis de marcha para cada paciente

In [ ]:
gait = build_gait_old()  # misma configuracion que en Script2.ipynb

all_summaries = []
processed = 0

for patient_id, file_path in trunk_files.items():
    if processed >= MAX_PATIENTS:
        break

    print(f"\n[{patient_id}] procesando {os.path.basename(file_path)}")
    summary_df = run_gait_freeliving(
        gait_model=gait,
        patient_id=patient_id,
        file_path=file_path,
        output_dir=OUTPUT_DIR,
        group=GROUP,
        height_m=PATIENT_HEIGHT_M,
        fs=None,
    )
    if summary_df is not None:
        all_summaries.append(summary_df)
        processed += 1

print(f"\n{processed} paciente(s) procesado(s)")


## 6. Combinar todos los resumenes en un unico CSV

In [ ]:
if all_summaries:
    combined = pd.concat(all_summaries, ignore_index=True)
    combined_path = os.path.join(OUTPUT_DIR, "ALL_patients_freeliving_summary.csv")
    combined.to_csv(combined_path, index=False)
    print(f"Resumen combinado guardado -> {combined_path}")
    display(combined)
else:
    print("No se genero ningun resumen.")
